In [ ]:
import pandas as pd 
import numpy as np 
from pathlib import Path
import umap 
import plotly.express as px

from sklearn.metrics.pairwise import cosine_similarity

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
BASE_DIR = PROJECT_ROOT
PROCESSED_DIR = BASE_DIR / "data" / "processed"

song_embeddings = pd.read_parquet(
    PROCESSED_DIR / "song_embeddings.parquet"
)

song_embeddings.shape

In [ ]:
embedding_cols = [
    col for col in song_embeddings.columns
    if col.startswith("embedding_")
]

X = song_embeddings[embedding_cols].values

X.shape

In [ ]:
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2, 
    metric="cosine",
    random_state=42
)

coords = reducer.fit_transform(X)

coords.shape

In [ ]:
song_atlas = song_embeddings[[
    "track_id", 
    "track_name",
    "album_name",
    "release_year"
]].copy()

song_atlas["x"] = coords[:, 0]
song_atlas["y"] = coords[:, 1]

song_atlas.head()

In [ ]:
fig = px.scatter(
    song_atlas,
    x="x",
    y="y",
    color="album_name",
    hover_data=["track_name", "album_name", "release_year"],
    title="BTS Song Atlas — Lyric Embedding Map"
)

fig.show()

In [ ]:
fig = px.scatter(
    song_atlas, 
    x="x",
    y="y",
    color="album_name",
    hover_name="track_name",
    hover_data=["release_year"],
    width=1200,
    height=800,
    title="BTS Song Atlas"
)

fig.update_traces(marker=dict(size=10))

fig.show()

## Similarity Search

In [ ]:
embedding_cols = [
    col for col in song_embeddings.columns
    if col.startswith("embedding_")
]

embedding_matrix = song_embeddings[embedding_cols].values

In [ ]:
import re

def canonical_title(title): 
    title = title.lower()
    title = re.sub(r"\s*-\s*live$", "", title)
    title = re.sub(r"\s*-\s*japanese ver\.$", "", title)
    title = re.sub(r"\s*\(.*?\)", "", title)
    title = re.sub(r"[^a-z0-9가-힣\s]", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


In [ ]:
def find_similar_songs(song_title, top_n=10, deduplicate=True):
    matches = song_embeddings[
        song_embeddings["track_name"].str.lower() == song_title.lower()
    ]

    if matches.empty:
        print(f"No exact match found for: {song_title}")
        return None

    target_idx = matches.index[0]
    target_vector = embedding_matrix[target_idx].reshape(1, -1)

    similarities = cosine_similarity(target_vector, embedding_matrix)[0]

    results = song_embeddings[[
        "track_id",
        "track_name",
        "album_name",
        "release_year"
    ]].copy()

    results["similarity"] = similarities
    results["canonical_title"] = results["track_name"].apply(canonical_title)

    target_canonical = canonical_title(song_title)

    if deduplicate:
        results = results.sort_values("similarity", ascending=False)
        results = results.drop_duplicates(subset=["canonical_title"], keep="first")

    target_clean = canonical_title(song_title)

    results = results[
        results["canonical_title"] != target_clean
    ]

    results = results[
        ~results["track_name"].str.lower().str.contains(song_title.lower(), regex=False)
    ]

    return results.sort_values("similarity", ascending=False).head(top_n)

In [ ]:
find_similar_songs("So What", top_n=10)

In [ ]:
find_similar_songs("Dimple", top_n=10)

In [ ]:
find_similar_songs("Boy in Luv", top_n=10)

In [ ]:
find_similar_songs("Black Swan", top_n=10)

In [ ]:
# TODO:
# Improve recommendations by combining:
# - lyric embeddings
# - album context
# - release year
# - audio features (Spotify)
# - clustering

## Section 4 — HDBSCAN Semantic clustering

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=1,
    metric="euclidean",
    prediction_data=True
)

clusters = clusterer.fit_predict(embedding_matrix)

clusters

In [ ]:
song_atlas["cluster"] = clusters

song_atlas.head()

In [ ]:
song_atlas["cluster"].value_counts().sort_index()


In [ ]:
n_clusters = len(song_atlas["cluster"].unique()) - (
    -1 in song_atlas["cluster"].unique()
)

print(f"Clusters found: {n_clusters}")

In [ ]:
fig = px.scatter(
    song_atlas,
    x="x",
    y="y",
    color="cluster",
    hover_name="track_name",
    hover_data=["album_name", "release_year"],
    width=1200,
    height=800, 
    title="BTS Song Atlas - Semantic Clusters"
)

fig.update_traces(marker=dict(size=10))

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    xaxis_showticklabels=False,
    yaxis_showticklabels=False
)

fig.show()

In [ ]:
for cluster in sorted(song_atlas["cluster"].unique()):

    print("=" * 60)
    print(f"Cluster {cluster}")

display(
    song_atlas[
        song_atlas["cluster"] == cluster
    ][[
        "track_name",
        "album_name",
        "release_year"
    ]].head(20)
)

In [ ]:
song_atlas["cluster"].value_counts().sort_index()

## Section 5 — K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
scores = []

for k in range(5, 16): 
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    labels = model.fit_predict(embedding_matrix)

    score = silhouette_score(embedding_matrix, labels)

    scores.append((k, score))

pd.DataFrame(scores, columns=["k", "silhouette_score"])


In [ ]:
song_master = pd.read_csv(PROCESSED_DIR / "song_master.csv")

atlas_export = song_atlas.merge(
    song_master[[
        "track_id", 
        "spotify_url",
        "image_url",
        "lyrics_clean",
        "word_count",
        "character_count"
    ]], 
    on="track_id",
    how="left"
)

# Full metadata export
atlas_export.to_csv(
    PROCESSED_DIR / "song_atlas_full.csv",
    index=False
)

# App map export: only songs with embeddings/coordinates/lyrics
atlas_app = atlas_export[
    atlas_export["lyrics_clean"].notna()
].copy()

atlas_app.to_csv(
    PROCESSED_DIR / "song_atlas.csv",
    index=False
)

print("Full:", atlas_export.shape)
print("App:", atlas_app.shape)

In [ ]:
check = pd.read_csv(PROCESSED_DIR / "song_atlas.csv")

print(check.shape)
print(check.isna().sum().sort_values(ascending=False))
check.head()